# RIS-ISAC Beamforming Demo

Demonstration of SNR/CRB-Constrained Joint Beamforming and Reflection Designs.

**Reference**: Rang Liu et al., IEEE TWC 2024, arXiv:2301.11134

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src import RIS_ISAC_System, AlternatingOptimizationSolver
from src.channel_model import RISChannelModel

## 1. System Setup (Table I Parameters)

In [ ]:
# Table I parameters
M = 4       # BS antennas
K = 2       # Users
L = 30      # RIS elements
P_max = 10e-3  # 10 mW
noise_power = 3.98e-12  # mW
sinr_thresh_dB = 10

system = RIS_ISAC_System(M=M, K=K, L=L, P_max=P_max, 
                         noise_power=noise_power,
                         sinr_thresh_dB=sinr_thresh_dB, seed=42)

print(f"System: M={M} BS antennas, K={K} users, L={L} RIS elements")
print(f"P_max = {P_max*1e3:.1f} mW, SINR threshold = {sinr_thresh_dB} dB")
print(f"RIS phases: |θ_l| = {np.mean(np.abs(system.theta)):.6f} (should be 1.0)")

## 2. SNR-Constrained Solution (Problem P1)

In [ ]:
solver_snr = AlternatingOptimizationSolver(
    system, problem_type='snr', snr_min_dB=5.0, max_iter=30
)

result_snr = solver_snr.solve()

print(f"SNR-Constrained Solution:")
print(f"  Sum rate: {result_snr['sum_rate']:.4f} bps/Hz")
print(f"  Radar SNR: {10*np.log10(result_snr['snr_sensing'] + 1e-20):.1f} dB")
print(f"  Converged: {result_snr['converged']}")
print(f"  Iterations: {result_snr['iterations']}")

## 3. CRB-Constrained Solution (Problem P2)

In [ ]:
system2 = RIS_ISAC_System(M=M, K=K, L=L, P_max=P_max,
                          noise_power=noise_power,
                          sinr_thresh_dB=sinr_thresh_dB, seed=42)

solver_crb = AlternatingOptimizationSolver(
    system2, problem_type='crb', crb_max=1e-2, max_iter=30
)

result_crb = solver_crb.solve()

print(f"CRB-Constrained Solution:")
print(f"  Sum rate: {result_crb['sum_rate']:.4f} bps/Hz")
print(f"  CRB: {result_crb['crb']:.6e}")
print(f"  Converged: {result_crb['converged']}")
print(f"  Iterations: {result_crb['iterations']}")

## 4. Convergence Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# SNR-constrained convergence
axes[0].plot(result_snr['history'], 'b-o', markersize=4)
axes[0].set_xlabel('AO Iteration')
axes[0].set_ylabel('Sum Rate (bps/Hz)')
axes[0].set_title('SNR-Constrained AO Convergence')
axes[0].grid(True, alpha=0.3)

# CRB-constrained convergence
axes[1].plot(result_crb['history'], 'r-o', markersize=4)
axes[1].set_xlabel('AO Iteration')
axes[1].set_ylabel('Sum Rate (bps/Hz)')
axes[1].set_title('CRB-Constrained AO Convergence')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('convergence.png', dpi=150)
plt.show()

## 5. RIS Elements Sensitivity

In [ ]:
L_values = [10, 20, 30, 40, 50]
rates_snr = []
rates_crb = []

for L_val in L_values:
    sys_s = RIS_ISAC_System(M=M, K=K, L=L_val, seed=42)
    sol_s = AlternatingOptimizationSolver(sys_s, problem_type='snr', 
                                          snr_min_dB=5.0, max_iter=20)
    res_s = sol_s.solve()
    rates_snr.append(res_s['sum_rate'])
    
    sys_c = RIS_ISAC_System(M=M, K=K, L=L_val, seed=42)
    sol_c = AlternatingOptimizationSolver(sys_c, problem_type='crb',
                                          crb_max=1e-2, max_iter=20)
    res_c = sol_c.solve()
    rates_crb.append(res_c['sum_rate'])

plt.figure(figsize=(8, 5))
plt.plot(L_values, rates_snr, 'b-o', label='SNR-Constrained')
plt.plot(L_values, rates_crb, 'r-s', label='CRB-Constrained')
plt.xlabel('Number of RIS Elements (L)')
plt.ylabel('Sum Rate (bps/Hz)')
plt.title('Sum Rate vs. RIS Elements')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('ris_elements.png', dpi=150)
plt.show()